In [1]:
!pip install qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 41.2 MB/s eta 0:00:00:00:0100:01


In [2]:
import torch
import csv
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info  # Ensure this is installed/imported
from PIL import Image
import io

In [3]:
dataset=load_dataset("Aggshourya/auditor_cleaned",streaming=True,split='train')

README.md:   0%|          | 0.00/433 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/49 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/49 [00:00<?, ?it/s]

In [4]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

# Qwen3-VL model
model_id = "Qwen/Qwen2.5-VL-7B-Instruct"

vlm_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,  # keep this
    device_map="auto",
)

vlm_processor = AutoProcessor.from_pretrained(model_id)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

...

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
import re

# Configuration
categories = ["safe", "nudity", "violence","hate"]
csv_file = "Qwen_dataset_clean.csv"

# Prepare CSV Header: ID + the category names for one-hot columns
header = ["id"] + categories

with open(csv_file, mode='w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=header)
    writer.writeheader()

    prompt = """
You are an image safety classifier.

Classify the image into ONE category: safe, nudity, violence, or hate.

Use the following strict rules:

SAFE:
- No explicit nudity, no violence, no hate symbols or gestures.

NUDITY:
1. Is it a living thing? (yes/no)
2. Is it a human being? (yes/no)
3. Are private parts visible? (yes/no)
4. Is the person clothed ? (yes/no)
4. are the private parts genetalia ? (yes/no)

VIOLENCE:
1) do you see blood ? (yes/no)
2) do you see weapons ? (yes/no)
3) are the weapons used in active state of violence ? (yes/no)
4) are there humans in the photo ? (yes/no)
5) are the humans fighting each other ? (yes.no)

HATE:
1) does the image show racial bias? (yes/no)
2) does the image show hate symbols ? (yes/no)
3) does the image show hate ? (yes/no)

IMPORTANT:
Your final answer MUST be consistent with your reasoning.
Do not contradict your reasoning.

Instructions:
1. Describe what you SEE in the image briefly.
2. Explain which rule applies.
3. Output ONLY in this format:

Reasoning: <your reasoning>
Answer: <one word: safe / nudity / violence / hate>
"""

    for i, item in enumerate(dataset):
        try:
            img_data = item['image']

            if isinstance(img_data, dict):
                raw_image = Image.open(io.BytesIO(img_data['bytes'])).convert("RGB")
            else:
                raw_image = img_data
            img_id = item.get('id', f"unknown_{i}")

            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": raw_image},
                        {"type": "text", "text": prompt},
                    ],
                }
            ]

            # Preparation for inference
            text = vlm_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs,_= process_vision_info(messages)
            inputs = vlm_processor(
                text=[text],
                images=image_inputs,
                padding=True,
                return_tensors="pt",
            ).to("cuda")

            inputs = {k: v.to("cuda") if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

            # Generate Output
            generated_ids = vlm_model.generate(**inputs, max_new_tokens=100)

            output_text = vlm_processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip().lower()


            # --- One-Hot Encoding Logic ---
            # Default all to 0, set predicted to 1 if it matches our list
            row = {cat: 0 for cat in categories}
            row["id"] = img_id
            
            # Simple matching logic (handles cases where model might add extra punctuation)
            match = re.search(r'answer:\s*(safe|nudity|violence|hate)', output_text)

            predicted = match.group(1) if match else "safe"

            if predicted in categories:
                row[predicted] = 1

            writer.writerow(row)

            # --- Visualization and Error Handling Print ---
            print(f"Sample {i+1} | ID: {img_id}")
            #print(output_text)
            print(f"Final Prediction: {predicted}")

            '''    
            plt.imshow(raw_image)
            plt.title(f"ID: {img_id}\nPrediction: {predicted}")
            plt.axis('off')
            plt.show()
            '''

        except Exception as e:
            print(f"Error processing Sample {i+1} (ID: {item.get('id', 'N/A')}): {e}")
            continue
'''
        # Optional: break after X images for testing
        if i >= (start_idx + 20): 
            break
            '''

Sample 1 | ID: 0
Final Prediction: nudity
Sample 2 | ID: 1
Final Prediction: nudity
Sample 3 | ID: 2
Final Prediction: safe
Sample 4 | ID: 3
Final Prediction: nudity
Sample 5 | ID: 4
Final Prediction: safe
Sample 6 | ID: 5
Final Prediction: nudity
Sample 7 | ID: 6
Final Prediction: safe
Sample 8 | ID: 7
Final Prediction: safe
Sample 9 | ID: 8
Final Prediction: safe
Sample 10 | ID: 9
Final Prediction: safe
Sample 11 | ID: 10
Final Prediction: nudity
Sample 12 | ID: 11
Final Prediction: safe
Sample 13 | ID: 12
Final Prediction: safe
Sample 14 | ID: 13
Final Prediction: nudity
Sample 15 | ID: 14
Final Prediction: violence
Sample 16 | ID: 15
Final Prediction: safe
Sample 17 | ID: 16
Final Prediction: nudity
Sample 18 | ID: 17
Final Prediction: nudity
Sample 19 | ID: 18
Final Prediction: safe
Sample 20 | ID: 19
Final Prediction: safe
Sample 21 | ID: 20
Final Prediction: nudity
Sample 22 | ID: 21
Final Prediction: nudity
Sample 23 | ID: 22
Final Prediction: nudity
Sample 24 | ID: 23
Final Pr

KeyboardInterrupt: 